# 06 – Build the RAG eval dataset

Builds the LangSmith eval dataset from scratch: 100-recipe sample → LLM-generated
questions (with ground-truth recipe ids) + numeric-constraint questions (with computed
ground truth). Re-running regenerates everything.

In [120]:
import instructor
import json
import pandas as pd
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from langsmith import Client
from dotenv import load_dotenv

load_dotenv("../../.env")

qdrant_client = QdrantClient(url="http://localhost:6333")
ls_client = Client()

SEED = 42
df = pd.read_parquet("../data/recipes_sample.parquet")

### Sub sampling for eval purposes
Instead of generating the eval dataset on the entire corpus, we pick 100 recipes and create a second collection.
The eval RAG pipeline must point at this collection to avoid false negative.

In [121]:
sample = df.sample(n=100, random_state=SEED)

recipes = [
    {"id": int(r.RecipeId), "text": r.text}
    for r in sample.itertuples(index=False)
]

In [124]:
from qdrant_client.models import Distance, Modifier, PayloadSchemaType, SparseVectorParams, VectorParams, PointStruct

COLLECTION_NAME = "Recipes-collection-01-hybrid"
EVAL_COLLECTION_NAME = "Recipes-collection-01-hybrid-eval-sample-100"
SAMPLE_IDS = [r["id"] for r in recipes]

records = qdrant_client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=SAMPLE_IDS,
    with_payload=True,
    with_vectors=True,
)
assert len(records) == len(SAMPLE_IDS), f"expected {len(SAMPLE_IDS)} points, got {len(records)}"

points = [PointStruct(id=r.id, vector=r.vector, payload=r.payload) for r in records]

if qdrant_client.collection_exists(EVAL_COLLECTION_NAME):
    qdrant_client.delete_collection(EVAL_COLLECTION_NAME)

qdrant_client.create_collection(
  collection_name=EVAL_COLLECTION_NAME,
  vectors_config={
    "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
  },
  sparse_vectors_config={
    "bm25": SparseVectorParams(modifier=Modifier.IDF)
  }
)

col_indexes = [
  { "name": "Calories", "type": PayloadSchemaType.FLOAT },
  { "name": "ProteinContent", "type": PayloadSchemaType.FLOAT },
  { "name": "CarbohydrateContent", "type": PayloadSchemaType.FLOAT },
  { "name": "FatContent", "type": PayloadSchemaType.FLOAT },
  { "name": "total_time_minutes", "type": PayloadSchemaType.INTEGER },
]

for c in col_indexes:
  qdrant_client.create_payload_index(
    collection_name=EVAL_COLLECTION_NAME,
    field_name=c["name"],
    field_schema=c["type"]
  )

UPLOAD_BATCH = 64
for start in range(0, len(points), UPLOAD_BATCH):
  qdrant_client.upsert(
    collection_name=EVAL_COLLECTION_NAME,
    points=points[start:start + UPLOAD_BATCH],
    wait=True,
  )
  print(f"upserted {min(start + UPLOAD_BATCH, len(points)):,} / {len(points):,}")

print(f"copied {qdrant_client.count(EVAL_COLLECTION_NAME).count} points into {EVAL_COLLECTION_NAME}")

upserted 64 / 100
upserted 100 / 100
copied 100 points into Recipes-collection-01-hybrid-eval-sample-100


In [123]:
print(recipes[42]["text"])

Butterhorn Rolls
This delicious dinner roll always brings praise.  From the Mississippi Valley chapter of the United States Regional Cookbook, Culinary Arts Institute of Chicago, 1947.
Ingredients: cake yeast sugar salt milk eggs flour butter
Instructions: Crumble yeast into a bowl; add sugar, salt, milk, and eggs. Mix well; add half of flour and beat well. Add melted butter and remainder of flour. Knead until smooth, cover and let rise in a warm place until doubled. Divide in half, roll each piece into a circle 1/4 inch thick. Butter, if you like and cut each piece into 16 pie shaped pieces. Roll each piece, beginning at the wide end towards the tip end, so that the tip is kept at an equal distance from each end of the roll. Arrange shaped rolls on a well-greased baking sheet, placing the tip underneath  the roll to prevent it from popping up and spoiling the shape of the roll. Allow rolls to rise until doubled and bake at 375F for 12 to 15 minutes. *To make crescent rolls, curve roll

### Generate questions with an LLM

Structured output via instructor. The prompt asks for three buckets (single / multi /
unanswerable) and enforces vocabulary realism: users haven't read the recipes, so
questions must paraphrase instead of echoing the recipes' wording.

In [ ]:
def build_eval_gen_prompt(n_total=60, n_single=15, n_multi=10, n_unanswerable=10):
  return f"""
You are helping build an evaluation dataset for a recipe recommendation RAG system.
The system retrieves recipes using semantic search over each recipe's full text (name, description, ingredients, instructions), then answers the user's question grounded only in the retrieved recipes.

You will be given a list of recipes, each with an "id" and its full "text".

Generate EXACTLY {n_total} distinct questions a home cook might ask this assistant, grounded in the provided recipes. Never repeat a question, and never produce two questions whose single ground-truth recipe is the same. Speak as a real user talking about recipes — never use the words "chunk", "context" or "document".

Vary how many recipes answer each question:
- at least {n_single} items: answerable by a single recipe. Phrase these the way a user who WANTS that kind of dish would search — by ingredients, cuisine, dietary constraint, technique, or occasion. NEVER name the recipe's title in the question. Include enough distinguishing attributes that this recipe is the single clear best match among the provided recipes; if several recipes would match equally, either add a distinguishing detail or list all of them in recipe_ids.
- at least {n_multi} items: require combining or comparing multiple recipes (e.g. "some vegetarian dinners that use mushrooms") — list ALL relevant recipe_ids.
- about {n_unanswerable}: plausible cooking questions that NONE of the provided recipes can answer (a dish, ingredient or cuisine absent from the list) — return an empty recipe_ids list and, in answer_example, state that nothing available fits.

Realism rules (apply to ALL questions):
- Write as someone who does NOT know which recipes exist. Never reference "your recipes", "your list", or a specific recipe name/title. No "in the X recipe" framing.
- Vary phrasing and length across the whole set. Aim for roughly this mix: ~40% "keyword" (terse search phrases like "vegan cookies no eggs"), ~40% "natural" (a real full-sentence question, e.g. "What can I make for a quick vegetarian dinner using mushrooms?"), ~20% "detailed" (a longer multi-attribute request). Do NOT make every question a keyword phrase.
- Set query_style to honestly describe how each question is phrased: "keyword", "natural", or "detailed". The label must match the actual wording.

Vocabulary realism — the most important rule:
- Real users have never read these recipes, so they cannot echo their wording. Do NOT copy distinctive phrases or rare terms verbatim from a recipe's text. Rephrase in everyday language: "granulated sugar" becomes "sugar", "saute" becomes "fry", a flowery dish description becomes how a shopper would casually describe it.
- Describe the NEED (occasion, craving, dietary constraint, what's in the fridge) rather than enumerating a recipe's ingredient list. Mention at most 2-3 ingredients per question, named the way people talk at the grocery store.
- Never stack 4 or more exact terms lifted from a single recipe: that makes retrieval artificially easy. One or two specific words (a cuisine, one distinctive ingredient) are fine when a real user would plausibly type them.
- After paraphrasing, re-check the ground truth: the target recipe(s) must still be identifiable as the best match(es) given only the question's wording. If paraphrasing makes the question ambiguous between several recipes, list all matching recipe_ids instead.

Rules:
- Ground truth must be correct given ONLY the provided recipes. Never invent recipes or ingredients.
- Base questions and answers on the recipe TEXT only. Do NOT ask about calories, nutrition or exact cooking time — those are covered by a separate, programmatically generated bucket.
- Prefer questions with a clear, checkable ground truth.
"""


SYSTEM_PROMPT = build_eval_gen_prompt()

In [ ]:
USER_PROMPT = f"Here is the list of recipes:\n{json.dumps(recipes, indent=2)}"

In [ ]:
from typing import Literal

class EvalItem(BaseModel):
  reasoning: str = Field(description="Reasoning why the question can be answered with the referenced recipes.")
  question: str = Field(description="Suggested question")
  query_style: Literal["keyword", "natural", "detailed"] = Field(description="How the question is phrased: terse keywords, a natural full-sentence question, or a detailed multi-attribute request.")
  recipe_ids: list[int] = Field(description="RecipeId(s) of the recipe(s) that answer the question (ground truth for retrieval).")
  answer_example: str = Field(description="Suggested answer grounded in the context")

In [ ]:
client = instructor.from_provider(
  "openai/gpt-5.4",
  mode=instructor.Mode.RESPONSES_TOOLS,
)

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    reasoning={"effort": "medium"},
    response_model=list[EvalItem],
)

In [ ]:
from collections import Counter


def _norm(q: str) -> str:
    return " ".join(q.lower().split())


# Dedupe: drop repeated questions and repeated single-recipe ground truths.
# (Multi-recipe items may legitimately reuse an id, so only dedupe single-recipe ones.)
seen_questions: set[str] = set()
seen_single_ids: set[int] = set()
eval_items = []
dropped = []
for item in response:
    qkey = _norm(item.question)
    single_id = item.recipe_ids[0] if len(item.recipe_ids) == 1 else None
    if qkey in seen_questions or (single_id is not None and single_id in seen_single_ids):
        dropped.append(item)
        continue
    seen_questions.add(qkey)
    if single_id is not None:
        seen_single_ids.add(single_id)
    eval_items.append(item)

print(f"kept {len(eval_items)}, dropped {len(dropped)} duplicate(s)")

# Fail loudly if the run is malformed rather than silently saving a bad set.
bucket = Counter(
    "unanswerable" if len(i.recipe_ids) == 0 else "single" if len(i.recipe_ids) == 1 else "multi"
    for i in eval_items
)
assert bucket["single"] and bucket["multi"] and bucket["unanswerable"], f"empty bucket: {dict(bucket)}"
assert len({_norm(i.question) for i in eval_items}) == len(eval_items), "duplicate questions remain"

kept 60, dropped 0 duplicate(s)


In [ ]:
import os

os.makedirs("../data", exist_ok=True)
with open("../data/eval_dataset_raw.json", "w") as f:
    json.dump([item.model_dump() for item in eval_items], f, indent=2, ensure_ascii=False)

print(f"Saved {len(eval_items)} items to ../data/eval_dataset_raw.json")

Saved 60 items to ../data/eval_dataset_raw.json


In [ ]:
counts = Counter()
for item in eval_items:
    n = len(item.recipe_ids)
    if n == 0:
        counts["0 recipes"] += 1
    elif n == 1:
        counts["1 recipe"] += 1
    else:
        counts[">1 recipes"] += 1

for label in ["0 recipes", "1 recipe", ">1 recipes"]:
    print(f"{label}: {counts[label]}")
print(f"total: {len(eval_items)}")

print("\nquery_style:")
style_counts = Counter(item.query_style for item in eval_items)
for style in ["keyword", "natural", "detailed"]:
    print(f"{style}: {style_counts[style]}")

0 recipes: 10
1 recipe: 25
>1 recipes: 25
total: 60

query_style:
keyword: 24
natural: 23
detailed: 13


### Constraint bucket

Queries with numeric constraints (calories, time, protein). The ground truth is
**computed** from the sample's numeric columns, not LLM-generated: each item stores a
machine-readable `constraints` list, scored by the deterministic
`constraint_satisfaction` evaluator. `constraint_matching_ids` is saved for inspection
only.

Thresholds are chosen so each query matches a sensible slice of the 100-recipe sample;
the assert catches a threshold drifting to 0 matches if the sample changes.

In [ ]:
CONSTRAINT_ITEMS = [
  {
    "question": "light dessert under 200 calories",
    "query_style": "keyword",
    "constraints": [{"field": "Calories", "op": "lt", "value": 200}],
  },
  {
    "question": "What can I make for breakfast that has less than 300 calories?",
    "query_style": "natural",
    "constraints": [{"field": "Calories", "op": "lt", "value": 300}],
  },
  {
    "question": "dinner main dish below 500 kcal",
    "query_style": "keyword",
    "constraints": [{"field": "Calories", "op": "lt", "value": 500}],
  },
  {
    "question": "I only have 20 minutes, what can I cook that's ready in under 20 minutes?",
    "query_style": "natural",
    "constraints": [{"field": "total_time_minutes", "op": "lt", "value": 20}],
  },
  {
    "question": "something I can make in half an hour or less",
    "query_style": "natural",
    "constraints": [{"field": "total_time_minutes", "op": "lte", "value": 30}],
  },
  {
    "question": "high protein meal with at least 30 grams of protein",
    "query_style": "natural",
    "constraints": [{"field": "ProteinContent", "op": "gte", "value": 30}],
  },
  {
    "question": "protein-packed dish, 20g protein or more",
    "query_style": "keyword",
    "constraints": [{"field": "ProteinContent", "op": "gte", "value": 20}],
  },
  {
    "question": "hearty filling meal with more than 500 calories",
    "query_style": "natural",
    "constraints": [{"field": "Calories", "op": "gt", "value": 500}],
  },
  {
    "question": "slow cooked recipe that takes over 2 hours",
    "query_style": "natural",
    "constraints": [{"field": "total_time_minutes", "op": "gt", "value": 120}],
  },
  {
    "question": "low calorie snack, max 150 kcal",
    "query_style": "keyword",
    "constraints": [{"field": "Calories", "op": "lte", "value": 150}],
  },
  {
    "question": "quick lunch ready in under 30 minutes and below 400 calories",
    "query_style": "detailed",
    "constraints": [
      {"field": "total_time_minutes", "op": "lt", "value": 30},
      {"field": "Calories", "op": "lt", "value": 400},
    ],
  },
  {
    "question": "I want a high protein dinner, more than 20 grams of protein but under 600 calories",
    "query_style": "detailed",
    "constraints": [
      {"field": "ProteinContent", "op": "gt", "value": 20},
      {"field": "Calories", "op": "lt", "value": 600},
    ],
  },
]

OPS = {
  "lt": lambda col, v: col < v,
  "lte": lambda col, v: col <= v,
  "gt": lambda col, v: col > v,
  "gte": lambda col, v: col >= v,
}


def matching_ids(constraints):
  mask = pd.Series(True, index=sample.index)
  for c in constraints:
    col = sample[c["field"]]
    mask &= col.notna() & OPS[c["op"]](col, c["value"])
  return [int(x) for x in sample.loc[mask, "RecipeId"]]


for item in CONSTRAINT_ITEMS:
  item["matching_ids"] = matching_ids(item["constraints"])
  n = len(item["matching_ids"])
  assert n > 0, f"no recipe satisfies: {item['question']!r}"
  print(f"{n:3d} recipes satisfy: {item['question']!r}")

 33 recipes satisfy: 'light dessert under 200 calories'
 50 recipes satisfy: 'What can I make for breakfast that has less than 300 calories?'
 73 recipes satisfy: 'dinner main dish below 500 kcal'
 24 recipes satisfy: "I only have 20 minutes, what can I cook that's ready in under 20 minutes?"
 45 recipes satisfy: 'something I can make in half an hour or less'
 14 recipes satisfy: 'high protein meal with at least 30 grams of protein'
 32 recipes satisfy: 'protein-packed dish, 20g protein or more'
 27 recipes satisfy: 'hearty filling meal with more than 500 calories'
  9 recipes satisfy: 'slow cooked recipe that takes over 2 hours'
 18 recipes satisfy: 'low calorie snack, max 150 kcal'
 30 recipes satisfy: 'quick lunch ready in under 30 minutes and below 400 calories'
 20 recipes satisfy: 'I want a high protein dinner, more than 20 grams of protein but under 600 calories'


### Upload to LangSmith

Recreates the dataset and uploads both buckets, with `bucket` + `query_style` as
example metadata for slicing experiment results.

In [ ]:
def get_text(recipe_id: int) -> str:
  records = qdrant_client.retrieve(
    collection_name=EVAL_COLLECTION_NAME,
    ids=[recipe_id],
    with_payload=True,
  )
  if not records:
    return ""
  return records[0].payload["text"]

In [ ]:
# Drop and recreate: old experiments stay visible in LangSmith but can't be
# compared per-example with new ones (example ids change).
dataset_name = "ragu-evaluation-dataset"
if ls_client.has_dataset(dataset_name=dataset_name):
  ls_client.delete_dataset(dataset_name=dataset_name)

dataset = ls_client.create_dataset(
  dataset_name=dataset_name,
  description="RAG evaluation dataset"
)

In [ ]:
def bucket_for(recipe_ids: list[int]) -> str:
  if not recipe_ids:
    return "unanswerable"
  return "single" if len(recipe_ids) == 1 else "multi"


# bucket + query_style let experiment results be sliced by question type.
llm_examples = [
  {
    "inputs": {
      "question": item.question,
    },
    "outputs": {
      "ground_truth": item.answer_example,
      "reference_context_ids": item.recipe_ids,
      "reference_description": [get_text(id) for id in item.recipe_ids],
    },
    "metadata": {
      "bucket": bucket_for(item.recipe_ids),
      "query_style": item.query_style,
    },
  }
  for item in eval_items
]

constraint_examples = [
  {
    "inputs": {"question": item["question"]},
    "outputs": {
      # Empty gold ids: the ID-based metrics skip this bucket,
      # constraint_satisfaction scores it instead.
      "reference_context_ids": [],
      "constraints": item["constraints"],
      "constraint_matching_ids": item["matching_ids"],
    },
    "metadata": {"bucket": "constraint", "query_style": item["query_style"]},
  }
  for item in CONSTRAINT_ITEMS
]

ls_client.create_examples(
  dataset_id=dataset.id,
  examples=llm_examples + constraint_examples,
)
print(f"uploaded {len(llm_examples)} LLM examples + {len(constraint_examples)} constraint examples")

uploaded 60 LLM examples + 12 constraint examples
